# PO-33 to PICO-8 Pattern 1 Export (v11)

This version fixes the **SFX Bit-Packing** to remove random effects and sets a more accurate playback speed for 120 BPM.

In [13]:
import pandas as pd
from pico8_exporter import Pico8Exporter

BIN_FILE = 'my_data_v3.bin'
with open(BIN_FILE, 'rb') as f:
    data = f.read()

print(f"Loaded {len(data)} bytes.")

Loaded 678632 bytes.


### Step 1: Decompile Pattern 1
Extract the note events using the 4-byte block logic.

In [14]:
def phases_to_bytes(phases):
    bytes_out = []
    for i in range(0, 16, 4):
        p = phases[i:i+4]
        b = (p[3] << 6) | (p[2] << 4) | (p[1] << 2) | p[0]
        bytes_out.append(b)
    return bytes_out

def unpack_phases(data_bytes):
    phases = []
    for b in data_bytes:
        phases.append(b & 0x03)
        phases.append((b >> 2) & 0x03)
        phases.append((b >> 4) & 0x03)
        phases.append((b >> 6) & 0x03)
    return phases

patterns = []
block = data[0:256]
phases = unpack_phases(block)
for step in range(16):
    for voice in range(4):
        start = (step * 64) + (voice * 16)
        b = phases_to_bytes(phases[start : start + 16])
        if b[0] != 0xAA or b[2] != 0xAA:
            patterns.append({
                'step': step + 1, 
                'voice': voice + 1, 
                'pitch': b[1] >> 4, 
                'inst_guess': b[3] >> 4
            })

df_pat1 = pd.DataFrame(patterns)
display(df_pat1.head(20))

,step,voice,pitch,inst_guess
0,1,2,13,8
1,1,3,2,7
2,1,4,13,8
3,2,2,11,14
4,2,3,4,1
5,2,4,11,14
6,3,2,2,7
7,3,3,13,8
8,3,4,2,7
9,4,2,4,1


### Step 2: Export Clean Cartridge
Mapping Pattern 1 Voices to SFX 10-13.
**Speed 8** is approx 112 BPM (at 60 FPS) or 56 BPM (at 30 FPS). Adjust as needed.

In [9]:
melodies = []
SEQUENCE_SPEED = 8 # Adjust this for BPM. lower = faster.

for v in range(1, 5):
    v_notes = []
    for _, row in df_pat1[df_pat1['voice'] == v].iterrows():
        # Pitch + Offset (24 = 2 octaves)
        pitch = int(row['pitch']) + 24
        v_notes.append((int(row['step'])-1, pitch, (v-1)%8, 7))
    
    melodies.append({'sfx_id': 10+v-1, 'speed': SEQUENCE_SPEED, 'notes': v_notes})

HEADER_END = 0x1000
pad_size = (len(data) - HEADER_END) // 16
inst_configs = [{'sfx_id': i, 'start_byte': HEADER_END + i*pad_size, 'end_byte': HEADER_END + (i+1)*pad_size, 'speed': 1} for i in range(8)]

exporter = Pico8Exporter()
pcm = exporter.extract_pcm(data)
exporter.create_p8_cart(pcm, inst_configs, [{'pattern_id': 0, 'channels': [10, 11, 12, 13]}], melodies, 'pattern2_clean.p8')

print("pattern1_clean.p8 CREATED! Load and check sequencer for effects (should be 0).")

pattern1_clean.p8 CREATED! Load and check sequencer for effects (should be 0).
